# Phase 16 — Final Quality Audit and Efficiency Benchmark

This notebook reviews the saved active-generation audit. It does not train, tune, reload the IMDb test reviews, or rerun the expensive benchmark. The fixed run directory below preserves the measurements and quality summaries for `recovery-20261101T044109.101917Z`.


In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Image, Markdown, display


def find_project_root():
    candidates = [Path.cwd(), *Path.cwd().parents]
    for candidate in candidates:
        if (candidate / "outputs/models/active_generation.json").is_file():
            return candidate
    raise FileNotFoundError("Could not locate the OpenBench project root.")


PROJECT_ROOT = find_project_root()
RUN_ID = "imdb-efficiency-20261101T123321.918848Z"
OUTPUT_DIR = PROJECT_ROOT / "outputs/efficiency" / RUN_ID
if not OUTPUT_DIR.is_dir():
    raise FileNotFoundError(f"Saved Phase 16 output is missing: {OUTPUT_DIR}")

print(f"Project: {PROJECT_ROOT}")
print(f"Saved benchmark: {OUTPUT_DIR}")

## Load the active generation

Use the same manifest-based discovery and inference loaders as the project. CPU loading is sufficient to confirm that the active artifacts and preprocessing can be opened; device timing below is read from the saved run.


In [ ]:
from deeptext.experiments.phase_16_final_audit import load_active_predictors

active, predictors = load_active_predictors(PROJECT_ROOT, device="cpu")
metadata = json.loads((OUTPUT_DIR / "metadata.json").read_text(encoding="utf-8"))
assert active["generation_id"] == metadata["active_generation_id"]

model_names = {
    "logistic_regression": "TF-IDF + Logistic Regression",
    "textcnn": "TextCNN",
    "rnn": "Vanilla RNN",
    "lstm": "LSTM",
    "gru": "GRU",
}
print(f"Active generation: {active['generation_id']}")
print(f"Loaded CPU predictors: {', '.join(model_names[key] for key in predictors)}")
print(f"Artifact hashes unchanged: {metadata['source_artifacts_unchanged']}")
print(f"Active pointer unchanged: {metadata['active_generation_pointer_unchanged']}")

## Read saved quality and benchmark results

The report separates each dataset and split. Historical scores remain attached to their source run IDs; unknown historical model-generation IDs are left unspecified.


In [ ]:
metrics = json.loads((OUTPUT_DIR / "metrics.json").read_text(encoding="utf-8"))
benchmark = json.loads((OUTPUT_DIR / "benchmark.json").read_text(encoding="utf-8"))
comparison = json.loads((OUTPUT_DIR / "comparison.json").read_text(encoding="utf-8"))
quality_summary = json.loads(
    (OUTPUT_DIR / "model_quality_summary.json").read_text(encoding="utf-8")
)
loading_results = pd.read_csv(OUTPUT_DIR / "loading_times.csv")
latency_results = pd.read_csv(OUTPUT_DIR / "latency_summary.csv")
batch_results = pd.read_csv(OUTPUT_DIR / "batch_benchmark.csv")
memory_results = pd.read_csv(OUTPUT_DIR / "memory_summary.csv")
quality_efficiency = pd.read_csv(OUTPUT_DIR / "efficiency_quality_comparison.csv")

print("Models:", len(benchmark["models"]), "| quality tracks:", len(metrics["tracks"]))
print("Timing sample:", benchmark["sample"]["count"], "reviews")
print("Batch sizes:", benchmark["configuration"]["batch_sizes"])
print("PyTorch:", benchmark["software_versions"]["torch"])
print("GPU:", benchmark["hardware"]["cuda_device"])

## Quality results by separate track

These saved metrics describe different source datasets and evaluation roles. The table does not aggregate them into a combined score.


In [ ]:
quality_display = quality_efficiency[
    (quality_efficiency["latency_device"] == "cpu")
    & quality_efficiency["quality_available"].astype(bool)
][
    [
        "quality_track_id",
        "quality_dataset_id",
        "quality_split",
        "quality_source_run_id",
        "model_name",
        "evaluated_examples",
        "accuracy",
        "precision",
        "recall",
        "f1",
    ]
]
display(
    quality_display.sort_values(["quality_track_id", "model_name"]).reset_index(
        drop=True
    )
)
display(
    pd.DataFrame(
        [
            {
                "summary": key,
                "leaders": ", ".join(value.get("leaders", [])),
                "value": value.get("value"),
                "track": value.get("track_id"),
            }
            for key, value in quality_summary.items()
            if isinstance(value, dict) and "leaders" in value
        ]
    )
)

## Model loading and single-review latency

Artifact loading and cold first-prediction latency are shown separately from steady-state end-to-end and forward-only latency. Values are milliseconds except artifact loading and initialization, which are seconds.


In [ ]:
display(
    loading_results[
        [
            "model_name",
            "device",
            "available",
            "model_initialization_seconds",
            "artifact_loading_seconds",
            "first_prediction_cold_start_latency_ms",
            "artifact_size_bytes",
        ]
    ]
    .sort_values(["model_name", "device"])
    .reset_index(drop=True)
)
display(
    latency_results[
        [
            "model_name",
            "device",
            "available",
            "single_review_observations",
            "single_review_mean_ms",
            "single_review_median_ms",
            "single_review_p95_ms",
            "single_review_minimum_ms",
            "single_review_maximum_ms",
            "forward_only_median_ms",
            "forward_only_p95_ms",
        ]
    ]
    .sort_values(["model_name", "device"])
    .reset_index(drop=True)
)

## Batch throughput and scaling

Each point uses all 130 representative reviews, including the final partial batch where applicable. The plot separates CPU from CUDA; unavailable measurements are omitted and remain documented in the CSV.


In [ ]:
available_batches = batch_results[batch_results["available"].astype(bool)]
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)
for axis, device in zip(axes, ("cpu", "cuda"), strict=True):
    subset = available_batches[available_batches["device"] == device]
    for model_name, model_rows in subset.groupby("model_name"):
        ordered = model_rows.sort_values("batch_size")
        axis.plot(
            ordered["batch_size"],
            ordered["reviews_per_second"],
            marker="o",
            label=model_name,
        )
    axis.set(title=device.upper(), xlabel="Batch size", ylabel="Reviews per second")
    axis.grid(alpha=0.25)
    axis.legend(fontsize=8)
fig.suptitle("Saved batch throughput by device")
fig.tight_layout()
display(fig)
plt.close(fig)

## CPU/CUDA results and memory

Logistic Regression supports CPU inference only. Neural CUDA measurements use synchronization around timed operations. Process RSS includes Python and shared libraries, so treat it as a process-level observation rather than isolated model memory.


In [ ]:
device_comparison = available_batches.pivot_table(
    index=["model_name", "batch_size"],
    columns="device",
    values="reviews_per_second",
    aggfunc="first",
).reset_index()
display(device_comparison)
display(
    memory_results[
        [
            "model_name",
            "device",
            "available",
            "rss_before_load_bytes",
            "rss_after_load_bytes",
            "rss_peak_during_inference_bytes",
            "observed_rss_increase_bytes",
            "peak_gpu_allocated_bytes",
            "peak_gpu_reserved_bytes",
            "unavailable_reason",
        ]
    ]
    .sort_values(["model_name", "device"])
    .reset_index(drop=True)
)

## Quality versus efficiency

The two panels preserve separate validation and UCI test tracks. Validation quality describes selected candidate checkpoints before full-data refit; latency is from the active refitted artifacts, so this pairing is an architecture-level descriptive comparison, not a same-checkpoint measurement.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
plot_specs = [
    ("imdb_validation_recovery_selection", "accuracy", "IMDb validation accuracy"),
    ("uci_short_text_test_active_generation_saved", "f1", "UCI short-text test F1"),
]
for axis, (track_id, metric, title) in zip(axes, plot_specs, strict=True):
    subset = quality_efficiency[
        (quality_efficiency["quality_track_id"] == track_id)
        & (quality_efficiency["latency_device"] == "cpu")
        & quality_efficiency["quality_available"].astype(bool)
    ]
    for _, row in subset.iterrows():
        axis.scatter(row["single_review_median_ms"], row[metric])
        axis.annotate(
            row["model_name"],
            (row["single_review_median_ms"], row[metric]),
            fontsize=8,
            xytext=(4, 4),
            textcoords="offset points",
        )
    axis.set(
        title=title,
        xlabel="Active artifact CPU median end-to-end latency (ms)",
        ylabel=metric.upper(),
    )
    axis.grid(alpha=0.25)
fig.tight_layout()
display(fig)
plt.close(fig)

## Saved benchmark visualizations

The plots below are generated with the audit and stored alongside its machine-readable outputs.


In [ ]:
for plot_name in metadata["plot_files"]:
    plot_path = OUTPUT_DIR / "plots" / plot_name
    print(plot_name)
    display(Image(filename=str(plot_path)))

## Findings and output validation

The report summarizes track-specific leaders and practical efficiency trade-offs. No composite score is used, the official test set has prior project use, and the semantic challenge labels are author-intended and unadjudicated.


In [ ]:
from deeptext.experiments.phase_16_final_audit import validate_saved_output

validation_result = validate_saved_output(OUTPUT_DIR)
display(Markdown((OUTPUT_DIR / "report.md").read_text(encoding="utf-8")))
print(f"Saved output validation: {validation_result}")
pareto_note = quality_summary["efficiency_awards"]["quality_efficiency_pareto"][
    "caveat"
]
print(f"Pareto note: {pareto_note}")